# Gradient Descent — Math Intuition

Gradient descent is the **engine** behind almost every ML optimiser.
Understanding it deeply means you can reason about learning rates,
convergence, saddle points, and why modern optimisers like Adam work.

**What we cover:**
- Derivatives and the gradient (direction of steepest ascent)
- Why we go in the *negative* gradient direction
- Visualising descent on 1-D and 2-D cost surfaces
- Batch vs Mini-batch vs Stochastic gradient descent
- The learning rate dilemma

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import sys
sys.path.insert(0, '../..')
from src.utils import set_style

set_style()
np.random.seed(0)

## 1. Derivative — Rate of Change

For a function $f(\theta)$, the derivative $f'(\theta)$ tells us how fast $f$ changes
at a point. If $f'(\theta) > 0$, the function is rising; if $< 0$, falling.

**Gradient descent rule (1-D):**
$$\theta := \theta - \alpha \cdot f'(\theta)$$

Subtract the derivative × learning rate → move toward the minimum.

In [ ]:
# Simple 1-D example: f(θ) = θ²   →  f'(θ) = 2θ   →  minimum at θ = 0
def f(theta):  return theta ** 2
def df(theta): return 2 * theta

# Simulate descent
alpha   = 0.25
theta   = 3.5
history = [theta]

for _ in range(20):
    theta = theta - alpha * df(theta)
    history.append(theta)

# Plot
t_vals = np.linspace(-4, 4, 300)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_vals, f(t_vals), color='#6B7280', lw=2, label='f(θ) = θ²')

for i, th in enumerate(history[:-1]):
    alpha_arrow = 0.9 ** i      # fade arrows over time
    ax.annotate(
        '', xy=(history[i+1], f(history[i+1])),
        xytext=(th, f(th)),
        arrowprops=dict(arrowstyle='->', color='#2563EB', lw=1.5, alpha=alpha_arrow),
    )

ax.scatter([h for h in history], [f(h) for h in history],
           c=range(len(history)), cmap='Blues', zorder=5, s=60, label='θ steps')
ax.set_xlabel('θ')
ax.set_ylabel('f(θ)')
ax.set_title(f'Gradient Descent on f(θ) = θ²  (α={alpha})')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Final θ = {history[-1]:.6f}  (true minimum = 0)')

## 2. The Gradient in Multi-Dimensions

For a function $J(\theta_0, \theta_1, \ldots, \theta_n)$, the **gradient** is the vector of partial derivatives:

$$\nabla_\theta J = \begin{bmatrix} \frac{\partial J}{\partial \theta_0} \\ \frac{\partial J}{\partial \theta_1} \\ \vdots \\ \frac{\partial J}{\partial \theta_n} \end{bmatrix}$$

This vector points in the direction of **steepest ascent**.  
We subtract it to go downhill.

In [ ]:
# 2-D cost surface: J(θ0, θ1) = θ0² + 2θ1²
def J(t0, t1):   return t0 ** 2 + 2 * t1 ** 2
def dJ_t0(t0):   return 2 * t0
def dJ_t1(t1):   return 4 * t1

# Gradient descent path
alpha = 0.15
t0, t1 = 3.0, 2.5
path = [(t0, t1)]

for _ in range(40):
    t0 = t0 - alpha * dJ_t0(t0)
    t1 = t1 - alpha * dJ_t1(t1)
    path.append((t0, t1))

path = np.array(path)

# Contour plot
t0_grid = np.linspace(-4, 4, 200)
t1_grid = np.linspace(-3, 3, 200)
T0, T1  = np.meshgrid(t0_grid, t1_grid)
Z       = J(T0, T1)

fig, ax = plt.subplots(figsize=(9, 7))
cp = ax.contourf(T0, T1, Z, levels=30, cmap='Blues', alpha=0.75)
ax.contour(T0, T1, Z, levels=15, colors='white', linewidths=0.5, alpha=0.5)
fig.colorbar(cp, ax=ax, label='J(θ₀, θ₁)')

ax.plot(path[:, 0], path[:, 1], 'o-', color='#DC2626', lw=2, ms=4, label='GD path')
ax.plot(*path[0],  'o', color='#16A34A', ms=10, label='Start')
ax.plot(*path[-1], '*', color='#F59E0B', ms=14, label='End')

ax.set_xlabel('θ₀')
ax.set_ylabel('θ₁')
ax.set_title('Gradient Descent on 2-D Cost Surface')
ax.legend()
plt.tight_layout()
plt.show()

## 3. The Learning Rate Dilemma

| Learning rate | Behaviour |
|---------------|-----------|
| Too large | Overshoots, oscillates, may diverge |
| Too small | Converges but takes many iterations |
| Just right | Smooth convergence |

In [ ]:
def run_gd(alpha, n=50, start=3.5):
    theta = start
    hist  = [theta]
    for _ in range(n):
        theta = theta - alpha * df(theta)
        hist.append(theta)
    return hist

configs = [
    (0.05,  '#6B7280', 'α=0.05 (too slow)'),
    (0.25,  '#2563EB', 'α=0.25 (good)'),
    (0.95,  '#16A34A', 'α=0.95 (on edge)'),
    (1.05,  '#DC2626', 'α=1.05 (diverges)'),
]

fig, ax = plt.subplots(figsize=(10, 5))
t_vals = np.linspace(-4, 4, 300)
ax.plot(t_vals, f(t_vals), 'k-', lw=1.5, alpha=0.3, label='f(θ) = θ²')

for alpha, color, label in configs:
    hist = run_gd(alpha, n=30)
    ax.plot(hist, [f(h) for h in hist], 'o-', color=color, lw=1.5, ms=4, label=label)

ax.set_xlabel('θ')
ax.set_ylabel('f(θ)')
ax.set_title('Effect of Learning Rate on Gradient Descent')
ax.set_xlim(-5, 5)
ax.set_ylim(-0.5, 20)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Batch vs Mini-Batch vs SGD

| Variant | Gradient computed on | Pros | Cons |
|---------|---------------------|------|------|
| **Batch GD** | All $m$ samples | Stable gradient | Slow on large data |
| **Mini-Batch GD** | $b$ samples (e.g. 32) | Balance of speed & stability | Requires batch size tuning |
| **SGD** | 1 sample | Very fast iteration | Noisy; may not converge smoothly |

In practice: **mini-batch GD** (b=32–512) is the default for neural networks.

In [ ]:
# Simulate noisy vs smooth cost curves
num_iter = 100

batch_cost = np.exp(-np.linspace(0, 3, num_iter)) * 10 + 0.5
sgd_cost   = batch_cost + np.random.randn(num_iter) * 1.2
mini_cost  = batch_cost + np.random.randn(num_iter) * 0.4

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(batch_cost, color='#2563EB', lw=2,   label='Batch GD (smooth)')
ax.plot(mini_cost,  color='#16A34A', lw=1.5, label='Mini-Batch GD')
ax.plot(sgd_cost,   color='#DC2626', lw=1,   alpha=0.8, label='SGD (noisy)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cost J(θ)')
ax.set_title('Batch vs Mini-Batch vs SGD — Cost Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Geometric Intuition: Gradient as a Hill

Imagine you're blindfolded on a hilly landscape. Your goal is to reach the valley (minimum cost).
At each step you:
1. Feel the slope under your feet (compute the gradient)
2. Take a small step in the **downhill** direction (subtract $\alpha \nabla J$)
3. Repeat until you stop moving (convergence)

**Convergence criterion:** Stop when $\|\nabla J\| < \varepsilon$ (gradient is nearly zero) or cost change is below a threshold.

In [ ]:
# Show convergence via gradient norm
def f_multi(theta):
    return theta[0]**2 + 0.5 * theta[1]**2

def grad_f(theta):
    return np.array([2 * theta[0], theta[1]])

alpha = 0.2
theta = np.array([3.0, 2.5])
grad_norms = []

for _ in range(50):
    g = grad_f(theta)
    grad_norms.append(np.linalg.norm(g))
    theta = theta - alpha * g

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grad_norms, color='#2563EB', lw=2)
ax.axhline(0.01, color='#DC2626', lw=1.2, linestyle='--', label='Convergence threshold ε=0.01')
ax.set_xlabel('Iteration')
ax.set_ylabel('||∇J||  (gradient norm)')
ax.set_title('Gradient Norm Decreasing → Convergence')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Concept | Key idea |
|---------|----------|
| Gradient $\nabla_\theta J$ | Vector pointing uphill |
| Update rule | $\theta := \theta - \alpha \nabla J$ (go downhill) |
| Learning rate $\alpha$ | Controls step size — tune carefully |
| Convergence | Gradient norm → 0 |
| Batch / SGD | Trade-off between stability and speed |

**Next:** [Implement gradient descent variants from scratch →](02_implementation_from_scratch.ipynb)